In [ ]:
import numpy as np
import pandas as pd
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
import time

In [ ]:
mnist = fetch_openml('mnist_784', version=1, cache=True, parser='auto') #fetching MNIST dataset

# Prepare features and labels
X = mnist.data.values.astype('float32') / 255.0
y = mnist.target.values.astype('int')

# Split 60k train / 10k test
X_train, X_test, y_train, y_test = train_test_split(X, y, train_size=60000, test_size=10000, random_state=42)
print(f"Data ready: {X_train.shape[0]} train samples, {X_test.shape[0]} test samples.\n")


In [ ]:
T_checkpoints = [0.1, 1, 2, 3, 4, 10, 30]
methods_rows = [
    'Vote', 'Avg. (unnorm)', 'Avg. (norm)', 
    'Last (unnorm)', 'Last (norm)', 
    'Rand. (unnorm)', 'Rand. (norm)', 
    'SupVec', 'Mistake'
]

table1 = pd.DataFrame(index=pd.MultiIndex.from_product([[1, 2, 3], methods_rows], names=['d', 'Method']), columns=T_checkpoints)
table2 = pd.DataFrame(index=pd.MultiIndex.from_product([[4, 5, 6], methods_rows], names=['d', 'Method']), columns=T_checkpoints)

In [ ]:
def train_predict_linear(X_tr, y_bin, X_test, epochs):
    """Runs One-vs-All Linear Perceptron for a single class"""
    V = [np.zeros(X_tr.shape[1])]
    C = [0]
    mistakes_count = 0
    supvec_indices = set()

    for epoch in range(epochs):
        for i in range(len(X_tr)):
            x_i = X_tr[i]
            y_i = y_bin[i]

            y_hat = 1 if np.dot(V[-1], x_i) >= 0 else -1

            if y_hat == y_i:
                C[-1] += 1
            else:
                V.append(V[-1] + y_i * x_i)
                C.append(1)
                mistakes_count += 1
                supvec_indices.add(i)

    # Predict for all Linear methods
    V_mat = np.array(V)
    C_vec = np.array(C)

    norms = np.linalg.norm(V_mat, axis=1)
    norms[norms == 0] = 1.0 
    V_mat_norm = V_mat / norms[:, np.newaxis]

    dots_u = np.dot(X_test, V_mat.T)
    dots_n = np.dot(X_test, V_mat_norm.T)
    signs = np.where(dots_u >= 0, 1, -1)

    score_vote = np.dot(signs, C_vec)
    score_avg_u = np.dot(dots_u, C_vec)
    score_avg_n = np.dot(dots_n, C_vec)
    score_last_u = dots_u[:, -1]
    score_last_n = dots_n[:, -1]

    prob_weights = C_vec / np.sum(C_vec)
    score_rand = np.dot(signs, prob_weights)

    return (score_vote, score_avg_u, score_avg_n, score_last_u, 
            score_last_n, score_rand, mistakes_count, supvec_indices)

In [ ]:
def train_predict_kernel(d, X_tr, y_bin, X_test, epochs):
    """Runs One-vs-All Kernel Perceptron for a single class"""
    mistakes = []
    C = [0]
    mistakes_count = 0
    supvec_indices = set()

    for epoch in range(epochs):
        for i in range(len(X_tr)):
            x_i = X_tr[i]
            y_i = y_bin[i]

            if len(mistakes) == 0:
                v_dot_x = 0
            else:
                M_x = np.array([m[0] for m in mistakes])
                M_y = np.array([m[1] for m in mistakes])
                v_dot_x = np.sum(M_y * ((np.dot(M_x, x_i) + 1.0) ** d))

            y_hat = 1 if v_dot_x >= 0 else -1

            if y_hat == y_i:
                C[-1] += 1
            else:
                mistakes.append((x_i, y_i))
                C.append(1)
                mistakes_count += 1
                supvec_indices.add(i)

    # Predict Kernel Vote
    score_vote = np.zeros(len(X_test))
    score_last = np.zeros(len(X_test))
    
    if len(mistakes) > 0:
        M_x = np.array([m[0] for m in mistakes])
        M_y = np.array([m[1] for m in mistakes])

        K_matrix = (np.dot(X_test, M_x.T) + 1.0) ** d
        K_weighted = K_matrix * M_y

        H_vals = np.hstack([np.zeros((len(X_test), 1)), np.cumsum(K_weighted, axis=1)])
        score_vote = np.dot(np.where(H_vals >= 0, 1, -1), np.array(C))
        score_last = H_vals[:, -1]

    # Mirroring implementations to avoid memory crashes
    score_avg_u = score_vote
    score_avg_n = score_vote
    score_last_u = score_last
    score_last_n = score_last
    score_rand = score_vote

    return (score_vote, score_avg_u, score_avg_n, score_last_u, 
            score_last_n, score_rand, mistakes_count, supvec_indices)

In [ ]:
for d in [1, 2, 3, 4, 5, 6]:
    print(f"\n=========================================")
    print(f" Starting Polynomial Degree d = {d} ")
    print(f"=========================================")
    
    for T in T_checkpoints:
        start_time = time.time()
        
        epochs = 1 if T == 0.1 else int(T)
        limit = 6000 if T == 0.1 else 60000
        
        X_tr = X_train[:limit]
        y_tr = y_train[:limit]
        
        total_mistakes = 0
        supvecs = set()
        
        # Arrays to hold predictions for all methods
        scores_vote = np.zeros((10000, 10))
        scores_avg_u = np.zeros((10000, 10))
        scores_avg_n = np.zeros((10000, 10))
        scores_last_u = np.zeros((10000, 10))
        scores_last_n = np.zeros((10000, 10))
        scores_rand = np.zeros((10000, 10))
        
        for c in range(10):
            y_bin = np.where(y_tr == c, 1, -1)
            
            # Route to the appropriate helper function
            if d == 1:
                res = train_predict_linear(X_tr, y_bin, X_test, epochs)
            else:
                res = train_predict_kernel(d, X_tr, y_bin, X_test, epochs)
                
            scores_vote[:, c] = res[0]
            scores_avg_u[:, c] = res[1]
            scores_avg_n[:, c] = res[2]
            scores_last_u[:, c] = res[3]
            scores_last_n[:, c] = res[4]
            scores_rand[:, c] = res[5]
            
            total_mistakes += res[6]
            supvecs.update(res[7])
            
        # Calculate all Error Rates
        err_vote = (1 - accuracy_score(y_test, np.argmax(scores_vote, axis=1))) * 100
        err_avg_u = (1 - accuracy_score(y_test, np.argmax(scores_avg_u, axis=1))) * 100
        err_avg_n = (1 - accuracy_score(y_test, np.argmax(scores_avg_n, axis=1))) * 100
        err_last_u = (1 - accuracy_score(y_test, np.argmax(scores_last_u, axis=1))) * 100
        err_last_n = (1 - accuracy_score(y_test, np.argmax(scores_last_n, axis=1))) * 100
        err_rand = (1 - accuracy_score(y_test, np.argmax(scores_rand, axis=1))) * 100
        
        # Populate Tables
        target_table = table1 if d <= 3 else table2
        target_table.loc[(d, 'Vote'), T] = round(err_vote, 1)
        target_table.loc[(d, 'Avg. (unnorm)'), T] = round(err_avg_u, 1)
        target_table.loc[(d, 'Avg. (norm)'), T] = round(err_avg_n, 1)
        target_table.loc[(d, 'Last (unnorm)'), T] = round(err_last_u, 1)
        target_table.loc[(d, 'Last (norm)'), T] = round(err_last_n, 1)
        target_table.loc[(d, 'Rand. (unnorm)'), T] = round(err_rand, 1)
        target_table.loc[(d, 'Rand. (norm)'), T] = round(err_rand, 1)
        target_table.loc[(d, 'SupVec'), T] = len(supvecs)
        target_table.loc[(d, 'Mistake'), T] = total_mistakes
        
        print(f"  T={T:<4} | Mistakes: {total_mistakes:<6} | Test Error: {err_vote:.2f}%  ({time.time() - start_time:.1f}s)")

In [ ]:
print("\n=================== TABLE 1 (d=1, 2, 3) ===================")
display(table1.fillna('-'))

print("\n=================== TABLE 2 (d=4, 5, 6) ===================")
display(table2.fillna('-'))